# CoT 파인튜닝 조종석 (train_cot.py 전용)

**목적**: 팀 고도화 프롬프트(`v4_story_cot`) + 기계 생성 CoT 응답으로 SFT — 설계: `PLAN_cot_finetune.md`
**판정**: 비교 대상은 **exp06_aug1_full (같은 1배 증강, shuffled 42.06%)** — +4%p 이상이면 CoT 승,
exp07(48.4%)까지 넘으면 aug2 확전(exp13). 핵심 지표는 항상 acc_shuffled.

**규칙**
- GPU 하나 = 다른 학습·추론 노트북과 동시 실행 금지 (Prompt_Experiments 커널이 모델을 잡고 있으면 먼저 재시작/종료)
- 본학습은 백그라운드 프로세스 — 커널이 죽어도 학습은 계속됨 (게이지만 다시 실행)
- ⚠️ **평가는 `--max-new-tokens 512` 필수** (기본 32면 CoT가 잘려 전부 파싱 실패) — 아래 eval_cmd에 내장됨
- ⚠️ 스모크에서 **peak VRAM** 확인 — target이 길어져 exp06/07(6.99GB)보다 오를 수 있음. 7.5GB 넘으면 중단하고 상의
- 밤샘: 전원 연결 + 덮개 열기 + 창 유지

In [1]:
# ① 실험 레지스트리 — CoT 실험 전용 (train_cot.py 사용, train.py와 분리)
DEFAULTS = dict(model="./models/Qwen3-VL-2B-Instruct", load_4bit=False,
                aug_mult=1, lr=1e-4, epochs=1, max_hours=0, prompt="v4_story_cot", extra="")

EXPERIMENTS = {
    "exp12_v4cot_aug1": dict(extra="--snapshot-steps 150"),   # 1배 증강 완주 (~11-13h) — exp06과 공정 비교
    # "exp13_v4cot_aug2": dict(aug_mult=2, extra="--snapshot-steps 150"),  # exp12 승리 시 확전 (~22-26h)
}

def cfg(name): return {**DEFAULTS, **EXPERIMENTS[name]}

def train_cmd(name, smoke=False):
    c = cfg(name)
    run = name + ("_smoke" if smoke else "")
    cmd = (f"python scripts/train_cot.py --run-name {run} --model {c['model']} "
           f"--aug-mult {c['aug_mult']} --lr {c['lr']} --epochs {c['epochs']} "
           f"--max-hours {c['max_hours']} --prompt {c['prompt']} {c['extra']}").strip()
    if c["load_4bit"]: cmd += " --load-4bit"
    if smoke: cmd += " --max-samples 12 --grad-accum 4 --max-steps 5"
    return cmd

def eval_cmd(name):
    c = cfg(name)
    cmd = (f"python scripts/eval_zero_shot.py --model {c['model']} "
           f"--adapter ./outputs/runs/{name}/adapter --prompt {c['prompt']} --max-new-tokens 512")
    if c["load_4bit"]: cmd += " --load-4bit"
    return cmd

for n in EXPERIMENTS: print(f"[{n}]\n  학습: {train_cmd(n)}\n  평가: {eval_cmd(n)}")

[exp12_v4cot_aug1]
  학습: python scripts/train_cot.py --run-name exp12_v4cot_aug1 --model ./models/Qwen3-VL-2B-Instruct --aug-mult 1 --lr 0.0001 --epochs 1 --max-hours 0 --prompt v4_story_cot --snapshot-steps 150
  평가: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs/exp12_v4cot_aug1/adapter --prompt v4_story_cot --max-new-tokens 512


In [2]:
# ② 실험 선택
NAME = "exp12_v4cot_aug1"

In [3]:
# ⓪ CoT target 미리보기 (GPU 불필요, ~30초) — 형식·분해 품질 눈검사
!python scripts/train_cot.py --run-name preview --preview 3

학습 제외: ./splits/holdout_300.csv (300개 누적)
문장 이벤트 분해 중 (spacy)...

===== Eomjsw =====
문장: A hand lowers a tool onto a towel, then moves to carefully work on the nails, and finally shifts away, leaving the tools behind.
--- target ---
[Story Analysis]
- Event 1: A hand lowers a tool onto a towel
- Event 2: moves to carefully work on the nails
- Event 3: shifts away
- Event 4: leaving the tools behind.
[Chronological Mapping]
- 1st: Image 3
- 2nd: Image 1
- 3rd: Image 2
- 4th: Image 4
[Final Answer]
<ANSWER>[3, 1, 2, 4]</ANSWER>

===== fm4Avz =====
문장:  We see a man throwing shingles into a trailer.
--- target ---
[Story Analysis]
- Event 1: We see
- Event 2: a man throwing shingles into a trailer.
[Chronological Mapping]
- 1st: Image 4
- 2nd: Image 2
- 3rd: Image 3
- 4th: Image 1
[Final Answer]
<ANSWER>[4, 2, 3, 1]</ANSWER>

===== 96NfXn =====
문장: The climber balances on the slackline with arms extended, then the scene shifts to a rock climber ascending a cliff before cutting to an inter

In [4]:
# ③ 스모크 (~3분) — 사이클 검증 + ⚠️ 종료 로그의 peak VRAM 확인 (7.5GB 초과 시 중단)
!{train_cmd(NAME, smoke=True)}

학습 제외: ./splits/holdout_300.csv (300개 누적)
문장 이벤트 분해 중 (spacy)...
기반 12개 x 증강 1 = 학습 항목 12개 | 평균 target 272자
trainable params: 6,422,528 || all params: 2,133,954,560 || trainable%: 0.3010
[최종] 어댑터 저장 -> ./outputs/runs\exp12_v4cot_aug1_smoke\adapter

종료(완주): 3 스텝, 12 항목, 0.01시간, peak VRAM 7.05GB
다음: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs\exp12_v4cot_aug1_smoke\adapter --prompt v4_story_cot --max-new-tokens 512



Loading weights: 100%|██████████| 625/625 [00:03<00:00, 167.97it/s]
W0715 21:13:16.991000 19968 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels

epoch 1/1:   0%|          | 0/12 [00:00<?, ?it/s][transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.

epoch 1/1: 100%|██████████| 12/12 [00:24<00:00,  2.01s/it, loss=0.788, vram=7.01]


In [5]:
# ④ 본학습 시작 — 백그라운드 프로세스 (커널을 막지 않음 → 아래 게이지 셀 동작)
import subprocess, os
os.makedirs(f"./outputs/runs/{NAME}", exist_ok=True)
console = open(f"./outputs/runs/{NAME}/console.log", "w", encoding="utf-8")
proc = subprocess.Popen(train_cmd(NAME), shell=True, stdout=console, stderr=subprocess.STDOUT,
                        env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(f"학습 시작: PID {proc.pid} | 콘솔 로그: outputs/runs/{NAME}/console.log")
print("중단하려면: 새 셀에서 proc.kill() (또는 작업관리자에서 위 PID 트리 종료)")

학습 시작: PID 7972 | 콘솔 로그: outputs/runs/exp12_v4cot_aug1/console.log
중단하려면: 새 셀에서 proc.kill() (또는 작업관리자에서 위 PID 트리 종료)


In [6]:
# ⑤ 진행 게이지 — 30초마다 갱신, 학습 종료 시 자동으로 다음 셀(평가)로 진행
import sys, time
from IPython.display import clear_output
sys.path.insert(0, "scripts")
from watch_train import read_status, render

run_dir = f"./outputs/runs/{NAME}"
p = globals().get("proc")
while True:
    try:
        c, last = read_status(run_dir)
        line = render(c, last)
        done = (p is not None and p.poll() is not None) or \
               (last is not None and last.opt_step >= c["total_opt_steps"])
    except FileNotFoundError:
        line, done = "시작 대기 중... (이벤트 분해 + 모델 로드에 2~4분)", False
    clear_output(wait=True)
    print(time.strftime("%H:%M:%S"), line)
    if done:
        tail = open(f"{run_dir}/console.log", encoding="utf-8", errors="replace").read().splitlines()
        print("\n학습 종료 — 콘솔 로그 마지막:\n" + "\n".join(tail[-3:]))
        break
    time.sleep(30)

01:51:47 [#############################-]  98.8%  9120/9232 항목 | loss 0.031 | 1.80초/항목 | 경과 273분, 남은 0.1시간 | 완료 예상 01:55 (기준) | VRAM 7.57GB

학습 종료 — 콘솔 로그 마지막:

종료(완주): 577 스텝, 9235 항목, 4.62시간, peak VRAM 7.57GB
다음: python scripts/eval_zero_shot.py --model ./models/Qwen3-VL-2B-Instruct --adapter ./outputs/runs\exp12_v4cot_aug1\adapter --prompt v4_story_cot --max-new-tokens 512


In [7]:
# ⑥ holdout 300 평가 (~40-75분: CoT 생성이 길다) — experiments.csv에 자동 누적
!{eval_cmd(NAME)}

평가셋 300개, 모델 ./models/Qwen3-VL-2B-Instruct, 4bit=False
어댑터 적용: ./outputs/runs/exp12_v4cot_aug1/adapter
모델 로드 완료
완료: 전체 0.430 | 섞인 샘플 0.377 | identity 0.708 | 샘플당 11.25초, VRAM 4.8GB
기록: ./outputs/experiments.csv | 전체 예측: ./outputs/preds/Qwen3-VL-2B-Instruct_exp12_v4cot_aug1_v4_story_cot.csv | 오답: ./outputs/errors_Qwen3-VL-2B-Instruct_exp12_v4cot_aug1_v4_story_cot.csv



Loading weights: 100%|██████████| 625/625 [00:01<00:00, 547.75it/s]
W0716 01:52:09.146000 5764 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels

100%|██████████| 300/300 [56:16<00:00, 11.25s/it]


## 분석 — exp06(같은 증강)과 비교 + CoT 응답 질적 확인

In [8]:
# ⑦ 비교표 + 세그먼트 분해 (vs exp06 42.06% / exp07 48.41%)
import pandas as pd, sys
sys.path.insert(0, "scripts")
import prompt_lab as lab

exp = pd.read_csv("./outputs/experiments.csv")
cols = ["timestamp", "model_id", "prompt", "accuracy", "acc_shuffled", "acc_identity", "parse_fail", "sec_per_sample"]
display(exp[[c for c in cols if c in exp.columns]].sort_values("acc_shuffled", na_position="first").tail(8))

# 세그먼트 분해 (Type-1/2/3) — 취약 세그먼트가 실제로 올랐는지가 CoT 가설의 핵심
hold = lab.load_holdout()
def seg_table(preds_path, label):
    p = pd.read_csv(preds_path)[["Id", "correct"]]
    m = hold.merge(p, on="Id"); sh = m[~m.No_ordering]
    t = sh.groupby("Partition")["correct"].agg(n="size", acc="mean").round(3)
    print(f"--- {label} (shuffled {len(sh)}) : 전체 {sh.correct.mean():.3f} ---"); display(t)

base = "./outputs/preds/Qwen3-VL-2B-Instruct"
seg_table(f"{base}_{NAME}_v4_story_cot.csv", NAME)
import os
for other, lbl in [(f"{base}_exp06_aug1_full.csv", "exp06(aug1, v1)"), (f"{base}_exp07_aug2_full.csv", "exp07(aug2, v1)")]:
    if os.path.exists(other): seg_table(other, lbl)

,timestamp,model_id,prompt,accuracy,acc_shuffled,acc_identity,parse_fail,sec_per_sample
3,2026-07-13 04:13,./models/Qwen3-VL-2B-Instruct,NaN,0.1400,NaN,NaN,3,0.69
4,2026-07-13 04:18,./models/Qwen3-VL-4B-Instruct,NaN,0.1633,NaN,NaN,5,1.14
7,2026-07-14 01:37,./models/Qwen3-VL-2B-Instruct,v3_cot,0.1600,0.0000,1.0000,297,7.96
6,2026-07-14 00:57,./models/Qwen3-VL-2B-Instruct,v2_temporal,0.1233,0.0159,0.6875,15,1.45
10,2026-07-16 02:48,./models/Qwen3-VL-2B-Instruct+exp12_v4cot_aug1,v4_story_cot,0.4300,0.3770,0.7083,0,11.25
8,2026-07-14 17:16,./models/Qwen3-VL-2B-Instruct+exp06_aug1_full,v1_list,0.4800,0.4206,0.7917,0,0.87
5,2026-07-14 00:54,./models/Qwen3-VL-2B-Instruct+exp01_aug2_lr1e4,v1_list,0.4900,0.4563,0.6667,0,1.58
9,2026-07-15 10:52,./models/Qwen3-VL-2B-Instruct+exp07_aug2_full,v1_list,0.5233,0.4841,0.7292,0,0.88


holdout 300개 | 취약 세그먼트 41개 (shuffled 기준 34개)
--- exp12_v4cot_aug1 (shuffled 252) : 전체 0.377 ---


,n,acc
Partition,,
Type-1,34,0.147
Type-2,164,0.439
Type-3,54,0.333


--- exp06(aug1, v1) (shuffled 252) : 전체 0.421 ---


,n,acc
Partition,,
Type-1,34,0.147
Type-2,164,0.500
Type-3,54,0.352


--- exp07(aug2, v1) (shuffled 252) : 전체 0.484 ---


,n,acc
Partition,,
Type-1,34,0.176
Type-2,164,0.585
Type-3,54,0.370


In [9]:
# ⑧ CoT 응답 질적 확인 — 모델이 실제로 이벤트 분해→매핑을 수행하는지 (베끼기 퇴화 감시)
import pandas as pd
raw = pd.read_csv(f"./outputs/preds/Qwen3-VL-2B-Instruct_{NAME}_v4_story_cot.csv")
print(f"파싱 실패: {(~raw.parsed).sum()} / {len(raw)}")
for _, r in raw.sample(3, random_state=0).iterrows():
    print(f"\n===== {r.Id} (correct={r.correct}) =====\n문장: {r.Sentence}\n--- 응답 ---\n{r.raw}")

파싱 실패: 0 / 300

===== 2hNse2 (correct=True) =====
문장: The baker prepares the cake base, then adds the filling, smooths it out, and finally checks the baked cake with a skewer.
--- 응답 ---
[Story Analysis]
- Event 1: The baker prepares the cake base
- Event 2: adds the filling
- Event 3: smooths it out
- Event 4: checks the baked cake with a skewer.
[Chronological Mapping]
- 1st: Image 3
- 2nd: Image 1
- 3rd: Image 2
- 4th: Image 4
[Final Answer]
<ANSWER>[3, 1, 2, 4]</ANSWER>

===== 060d6p (correct=True) =====
문장: The mixture is spread on bread, followed by a display of bacon and lettuce, then bacon and cheese are layered onto bread, and the sandwich is served with fries on a plate as the camera zooms out slightly.
--- 응답 ---
[Story Analysis]
- Event 1: The mixture is spread on bread, followed by a display of bacon and lettuce
- Event 2: bacon and cheese are layered onto bread
- Event 3: the sandwich is served with fries on a plate as the camera zooms out slightly.
[Chronological Mapping

## 판정·다음 단계
| 결과 (vs exp06 42.06%) | 행동 |
|---|---|
| +4%p 이상 | CoT 승 — exp13(aug2, ~22-26h) 확전. exp07(48.4%)도 넘으면 주력 교체 후보 |
| ±4%p 이내 | 노이즈 — [Visual Evidence]에 CLIP 유사쌍 추가한 2차 버전 검토 후 판단 |
| 하락 | CoT SFT 보류, 자원을 exp08(aug4)·힌트 SFT로 — 결과는 어느 쪽이든 EXPERIMENTS.md에 기록 |

비용 참고: 채택 시 test 제출 추론 819개 × 8~15초 ≈ 2~3.5h (규정 24h 내 여유)